# WR Receiving Yards Prediction — Career-Based RNN + Optuna

Unified pipeline for predicting NFL wide receiver receiving yards per game.

**Key improvements over previous work (RNN.ipynb):**
1. **Career-based sequences** — RNN sees a player's full career, not just one season
2. **Optuna hyperparameter optimization** — for MLP, RNN, and TCN architectures
3. **Deeper architectures** — multi-layer BiLSTM/GRU with multi-head attention
4. **TCN alternative** — Temporal Convolutional Networks
5. **Unified evaluation** — all models evaluated on identical test set
6. **Ensemble** — weighted combination of best models

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import os, json, gc, time, random
from pathlib import Path

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import PredefinedSplit, GridSearchCV

# Tree models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, losses, Model, Input
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Bidirectional,
    LayerNormalization, BatchNormalization,
    MultiHeadAttention, GlobalAveragePooling1D,
    Masking, Conv1D, Add, Activation, Flatten
)

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Plotting
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = (12, 5)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# GPU memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow {tf.__version__}')
print(f'Optuna {optuna.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 1. Data Loading

Load the WR play-by-play aggregated data (2015-2025).

In [ ]:
df = pd.read_csv('../data/fully combined/wr_all_weeks.csv')

# Extract week from game_id
df['week'] = df['game_id'].str.split('_').str[1].astype(int)
df = df.sort_values(['receiver_player_id', 'season', 'week']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Unique players: {df["receiver_player_id"].nunique()}')
print(f'Target stats (receiving_yards):')
print(df['receiving_yards'].describe().round(2))

---
## 2. Feature Engineering (Career-Based)

**Key changes from RNN.ipynb:**
- Rolling features computed across **entire career** (not per-season)
- **No roll3** features (redundant with lag1 + roll5)
- Added **momentum features** (trend = roll5 − lag1)
- Added **interaction features** (target_share × pregame_total, etc.)
- `is_new_season` and `weeks_since_last_game` signal season boundaries to the RNN

In [ ]:
work_df = df.copy()

# --- Rename WP columns ---
wp_rename_map = {
    'yards_wp_<25': 'yards_wp_less_than_25',
    'yards_wp_>75': 'yards_wp_greater_than_75',
    'receptions_wp_<25': 'receptions_wp_less_than_25',
    'receptions_wp_>75': 'receptions_wp_greater_than_75',
    'targets_wp_<25': 'targets_wp_less_than_25',
    'targets_wp_>75': 'targets_wp_greater_than_75',
}
work_df = work_df.rename(columns={k: v for k, v in wp_rename_map.items() if k in work_df.columns})

# --- Infer player team and detect changes ---
work_df['player_team_inferred'] = np.where(
    (work_df['home_team'] == work_df['defteam']) & (work_df['away_team'] != work_df['defteam']),
    work_df['away_team'],
    np.where(
        (work_df['away_team'] == work_df['defteam']) & (work_df['home_team'] != work_df['defteam']),
        work_df['home_team'], np.nan
    )
)

prev_team = work_df.groupby('receiver_player_id')['player_team_inferred'].shift(1)
work_df['team_changed'] = (
    work_df['player_team_inferred'].notna() & prev_team.notna()
    & (work_df['player_team_inferred'] != prev_team)
).astype(int)

prev_season = work_df.groupby('receiver_player_id')['season'].shift(1)
work_df['is_new_season'] = (
    prev_season.notna() & (work_df['season'] != prev_season)
).astype(int)

# --- Weeks since last game (CAREER-BASED, not season-based) ---
work_df['season_week_abs'] = (work_df['season'] - 2015) * 22 + work_df['week']
work_df['weeks_since_last_game'] = (
    work_df.groupby('receiver_player_id')['season_week_abs']
    .diff().fillna(1).clip(lower=1).astype(float)
)
work_df.drop(columns=['season_week_abs'], inplace=True)

print(f'team_changed: {work_df["team_changed"].sum()} rows')
print(f'is_new_season: {work_df["is_new_season"].sum()} rows')
print(f'weeks_since_last_game stats:')
print(work_df['weeks_since_last_game'].describe().round(2))

In [ ]:
# --- Previous-season career features ---
season_career = (
    work_df.groupby(['receiver_player_id', 'season'], as_index=False)
    .agg(
        avg_yards=('receiving_yards', 'mean'),
        avg_target_share=('target_share', 'mean'),
        avg_epa=('epa', 'mean'),
        avg_air_yard_share=('air_yard_share', 'mean'),
        avg_catch_rate=('catch_rate', 'mean'),
        games_played=('game_id', 'count')
    )
    .sort_values(['receiver_player_id', 'season'])
)

for c in ['avg_yards', 'avg_target_share', 'avg_epa',
          'avg_air_yard_share', 'avg_catch_rate', 'games_played']:
    season_career[f'{c}_last_season'] = (
        season_career.groupby('receiver_player_id')[c].shift(1)
    )

career_cols = [
    'avg_yards_last_season', 'avg_target_share_last_season',
    'avg_epa_last_season', 'avg_air_yard_share_last_season',
    'avg_catch_rate_last_season', 'games_played_last_season',
]
season_career = season_career[['receiver_player_id', 'season'] + career_cols]
work_df = work_df.merge(season_career, on=['receiver_player_id', 'season'], how='left')
work_df[career_cols] = work_df[career_cols].fillna(0)
print(f'Career features added: {career_cols}')

In [ ]:
# --- Lag1 and Roll5 features (CAREER-BASED, no roll3) ---
rolling_source_cols = [
    'targets', 'receptions', 'air_yards', 'yac', 'tds', 'epa', 'wpa', 'catch_rate',
    'avg_depth', 'adot', 'yac_per_reception', 'td_rate', 'explosive_plays', 'first_downs',
    'yards_per_target', 'team_pass_attempts', 'team_air_yards', 'team_epa', 'air_yard_share',
    'target_share', 'qb_completions', 'qb_attempts', 'qb_air_yards', 'qb_cpoe', 'qb_comp_pct',
    'avg_score_diff', 'trailing_pct', 'leading_pct', 'avg_quarter', 'success_rate',
    'big_play_rate', 'avg_start_yardline', 'red_zone_targets', 'end_zone_targets',
    'third_down_targets', 'fourth_down_targets', 'high_leverage_targets',
    'second_and_long_targets', 'third_and_medium_targets', 'wp_var', 'target_share_std',
    'reception_std', 'def_targets_dev', 'def_receptions_dev', 'def_yards_dev', 'def_tds_dev',
    'def_epa_dev', 'yards_Q1', 'yards_Q2', 'yards_Q3', 'yards_Q4',
    'receptions_Q1', 'receptions_Q2', 'receptions_Q3', 'receptions_Q4',
    'targets_Q1', 'targets_Q2', 'targets_Q3', 'targets_Q4',
    'lost_yards_due_to_penalty',
    'yards_wp_less_than_25', 'yards_wp_25_45', 'yards_wp_45_55',
    'yards_wp_55_75', 'yards_wp_greater_than_75',
    'receptions_wp_less_than_25', 'receptions_wp_25_45', 'receptions_wp_45_55',
    'receptions_wp_55_75', 'receptions_wp_greater_than_75',
    'targets_wp_less_than_25', 'targets_wp_25_45', 'targets_wp_45_55',
    'targets_wp_55_75', 'targets_wp_greater_than_75',
    'weeks_since_last_game',
]

available_roll_cols = [c for c in rolling_source_cols if c in work_df.columns]
print(f'Rolling source columns found: {len(available_roll_cols)}')

# CAREER-BASED groupby (across seasons, not within season)
grp = work_df.groupby('receiver_player_id', sort=False)
derived_cols = []

for col in available_roll_cols:
    work_df[f'{col}_lag1'] = grp[col].transform(lambda s: s.shift(1))
    work_df[f'{col}_roll5'] = grp[col].transform(
        lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
    )
    derived_cols.extend([f'{col}_lag1', f'{col}_roll5'])

# Drop raw in-game columns to prevent leakage
work_df = work_df.drop(columns=available_roll_cols)
print(f'Derived lag1+roll5 features: {len(derived_cols)}')

In [ ]:
# --- Momentum features: trend direction (lag1 vs roll5) ---
# Positive = recent performance above average = upward trend
momentum_sources = [
    'targets', 'receptions', 'air_yards', 'epa', 'catch_rate',
    'target_share', 'yards_per_target', 'air_yard_share',
]
momentum_cols = []
for col in momentum_sources:
    lag1_col = f'{col}_lag1'
    roll5_col = f'{col}_roll5'
    if lag1_col in work_df.columns and roll5_col in work_df.columns:
        mcol = f'{col}_momentum'
        work_df[mcol] = work_df[lag1_col] - work_df[roll5_col]
        momentum_cols.append(mcol)

print(f'Momentum features added: {len(momentum_cols)}')
print(momentum_cols)

In [ ]:
# --- Interaction features ---
interaction_cols = []

# target_share × pregame_total = expected targets volume
if 'target_share_lag1' in work_df.columns:
    work_df['target_volume_interaction'] = (
        work_df['target_share_lag1'] * work_df['pregame_total']
    )
    interaction_cols.append('target_volume_interaction')

# air_yard_share × team_pass_attempts = expected air yards
if 'air_yard_share_lag1' in work_df.columns and 'team_pass_attempts_lag1' in work_df.columns:
    work_df['air_yards_expected'] = (
        work_df['air_yard_share_lag1'] * work_df['team_pass_attempts_lag1']
    )
    interaction_cols.append('air_yards_expected')

# catch_rate × targets = expected receptions
if 'catch_rate_lag1' in work_df.columns and 'targets_lag1' in work_df.columns:
    work_df['expected_receptions'] = (
        work_df['catch_rate_lag1'] * work_df['targets_lag1']
    )
    interaction_cols.append('expected_receptions')

# EPA per target
if 'epa_lag1' in work_df.columns and 'targets_lag1' in work_df.columns:
    work_df['epa_per_target'] = (
        work_df['epa_lag1'] / work_df['targets_lag1'].replace(0, np.nan)
    ).fillna(0)
    interaction_cols.append('epa_per_target')

print(f'Interaction features added: {len(interaction_cols)}')
print(interaction_cols)

In [ ]:
# --- Assemble final feature list ---
pregame_features = [
    'pregame_spread', 'pregame_total', 'surface', 'is_dome', 'temp_f',
    'humidity_pct', 'wind_mph', 'is_rain', 'is_snow', 'is_clear',
    'season', 'week', 'team_changed', 'is_new_season',
]
missing_pregame = [c for c in pregame_features if c not in work_df.columns]
if missing_pregame:
    raise ValueError(f'Missing pre-game features: {missing_pregame}')

feature_columns = pregame_features + career_cols + derived_cols + momentum_cols + interaction_cols
print(f'\nTotal features: {len(feature_columns)}')

# Keep identifiers + target + features
target_col = 'receiving_yards'
id_cols = ['receiver_player_id', 'receiver_player_name', 'game_id', 'season', 'week']
model_df = work_df[id_cols + [target_col] + feature_columns].copy()
print(f'model_df shape: {model_df.shape}')

---
## 3. Filtering & Target Transformation

- Drop rows without any history (all lag1 = NaN)
- **Sqrt transformation** of receiving yards (better than log1p for this distribution)
- Continuous sample weights (heavier weight on high-yardage games)

In [ ]:
# Drop rows where ALL lag1 features are NaN (player's first game ever)
lag1_cols = [c for c in feature_columns if c.endswith('_lag1')]
no_history = model_df[lag1_cols].isna().all(axis=1)
rows_before = len(model_df)
model_df = model_df.loc[~no_history].copy()
print(f'Dropped {rows_before - len(model_df)} rows with no history')
print(f'Remaining: {len(model_df)} rows')

# Fill remaining NaNs in features with 0 (partial history for early career games)
model_df[feature_columns] = model_df[feature_columns].fillna(0)

# Sqrt target transformation
model_df['receiving_yards_sqrt'] = np.sqrt(model_df['receiving_yards'].clip(lower=0))

print(f'\nTarget (sqrt) stats:')
print(model_df['receiving_yards_sqrt'].describe().round(3))
print(f'\nNaN check: {model_df[feature_columns].isna().sum().sum()} NaNs remaining')

---
## 4. Temporal Split & Scaling

Strict chronological split:
- **Train:** 2015-2021
- **Validation:** 2022-2023
- **Test:** 2024-2025

StandardScaler fitted on train only.

In [ ]:
train_seasons = list(range(2015, 2022))
val_seasons = [2022, 2023]
test_seasons = [2024, 2025]

train_df = model_df[model_df['season'].isin(train_seasons)].copy()
val_df = model_df[model_df['season'].isin(val_seasons)].copy()
test_df = model_df[model_df['season'].isin(test_seasons)].copy()

print(f'Train: {len(train_df)} rows ({train_seasons[0]}-{train_seasons[-1]})')
print(f'Val:   {len(val_df)} rows ({val_seasons})')
print(f'Test:  {len(test_df)} rows ({test_seasons})')

# --- Flat features and targets ---
X_train_flat = train_df[feature_columns].values
X_val_flat = val_df[feature_columns].values
X_test_flat = test_df[feature_columns].values

y_train_sqrt = train_df['receiving_yards_sqrt'].values
y_val_sqrt = val_df['receiving_yards_sqrt'].values
y_test_sqrt = test_df['receiving_yards_sqrt'].values

y_train_orig = train_df['receiving_yards'].values
y_val_orig = val_df['receiving_yards'].values
y_test_orig = test_df['receiving_yards'].values

# --- Scaling (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_val_scaled = scaler.transform(X_val_flat)
X_test_scaled = scaler.transform(X_test_flat)

print(f'\nFeature matrix shapes:')
print(f'  X_train: {X_train_scaled.shape}')
print(f'  X_val:   {X_val_scaled.shape}')
print(f'  X_test:  {X_test_scaled.shape}')

# --- Continuous sample weights ---
mean_y_train = np.mean(np.clip(y_train_orig, 0, None))
sample_weights_train = 1.0 + np.sqrt(np.clip(y_train_orig, 0, None) / mean_y_train)
print(f'\nSample weight range: [{sample_weights_train.min():.2f}, {sample_weights_train.max():.2f}]')

---
## 5. Career-Based Sequence Building

**Key change:** Sequences span across season boundaries.
- The RNN sees a player's full career trajectory
- `is_new_season` and `weeks_since_last_game` signal season transitions
- Target is always the NEXT game after the sequence window
- Split is based on which temporal period the TARGET game falls in

Pre-build sequences for multiple lengths so Optuna can search over them.

In [ ]:
def build_career_sequences(model_df, feature_columns, scaler, seq_len):
    """
    Build sliding-window sequences across each player's full career.
    The target is the game immediately after the sequence window.
    """
    X_seqs, y_seqs_sqrt, y_seqs_orig, seasons, indices = [], [], [], [], []

    for pid, group in model_df.groupby('receiver_player_id'):
        group = group.sort_values(['season', 'week'])
        n = len(group)
        if n <= seq_len:
            continue

        # Scale features using the pre-fitted scaler
        feat_vals = group[feature_columns].values
        feat_scaled = scaler.transform(feat_vals)

        target_sqrt = group['receiving_yards_sqrt'].values
        target_orig = group['receiving_yards'].values
        season_vals = group['season'].values
        idx_vals = group.index.values

        for i in range(n - seq_len):
            X_seqs.append(feat_scaled[i:i + seq_len])
            y_seqs_sqrt.append(target_sqrt[i + seq_len])
            y_seqs_orig.append(target_orig[i + seq_len])
            seasons.append(season_vals[i + seq_len])
            indices.append(idx_vals[i + seq_len])

    return (
        np.array(X_seqs, dtype=np.float32),
        np.array(y_seqs_sqrt, dtype=np.float32),
        np.array(y_seqs_orig, dtype=np.float32),
        np.array(seasons),
        np.array(indices),
    )


def split_sequences(X, y_sqrt, y_orig, seasons, train_seasons, val_seasons, test_seasons):
    """Split pre-built sequences by target game season."""
    tr = np.isin(seasons, train_seasons)
    va = np.isin(seasons, val_seasons)
    te = np.isin(seasons, test_seasons)
    return {
        'train': (X[tr], y_sqrt[tr], y_orig[tr]),
        'val':   (X[va], y_sqrt[va], y_orig[va]),
        'test':  (X[te], y_sqrt[te], y_orig[te]),
    }

In [ ]:
# Pre-build sequences for candidate lengths
SEQ_LENGTHS = [8, 12, 16, 24]
seq_data = {}

for sl in SEQ_LENGTHS:
    t0 = time.time()
    X, y_sq, y_or, seas, idxs = build_career_sequences(
        model_df, feature_columns, scaler, sl
    )
    splits = split_sequences(X, y_sq, y_or, seas, train_seasons, val_seasons, test_seasons)
    seq_data[sl] = splits
    elapsed = time.time() - t0

    print(f'seq_len={sl:>2d}  |  '
          f'train={splits["train"][0].shape[0]:>5d}  '
          f'val={splits["val"][0].shape[0]:>5d}  '
          f'test={splits["test"][0].shape[0]:>5d}  '
          f'features={X.shape[2]}  |  {elapsed:.1f}s')

n_features = X.shape[2]
print(f'\nTotal features per timestep: {n_features}')

# Compute sample weights for each sequence length
seq_weights = {}
for sl in SEQ_LENGTHS:
    y_tr_orig = seq_data[sl]['train'][2]
    mean_y = np.mean(np.clip(y_tr_orig, 0, None))
    seq_weights[sl] = 1.0 + np.sqrt(np.clip(y_tr_orig, 0, None) / mean_y)

---
## 6. Tree Model Baselines (XGBoost, LightGBM, RandomForest)

Flat feature models for a fair baseline comparison on the same test set.
All models use sqrt target + continuous sample weights.

In [ ]:
def evaluate_model(y_true_orig, y_pred_sqrt):
    """Evaluate predictions: convert sqrt back to original scale."""
    y_pred_orig = np.clip(y_pred_sqrt, 0, None) ** 2
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    r2 = r2_score(y_true_orig, y_pred_orig)
    return mae, rmse, r2, y_pred_orig


results = {}

# --- XGBoost ---
xgb = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=0
)
xgb.fit(X_train_scaled, y_train_sqrt, sample_weight=sample_weights_train)
mae, rmse, r2, xgb_test_pred = evaluate_model(y_test_orig, xgb.predict(X_test_scaled))
results['XGBoost'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
xgb_val_pred = xgb.predict(X_val_scaled)
print(f'XGBoost      | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')

# --- LightGBM ---
lgb = LGBMRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, num_leaves=31, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=-1
)
lgb.fit(X_train_scaled, y_train_sqrt, sample_weight=sample_weights_train)
mae, rmse, r2, lgb_test_pred = evaluate_model(y_test_orig, lgb.predict(X_test_scaled))
results['LightGBM'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
lgb_val_pred = lgb.predict(X_val_scaled)
print(f'LightGBM     | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')

# --- RandomForest ---
rf = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=4,
    random_state=SEED, n_jobs=-1
)
rf.fit(X_train_scaled, y_train_sqrt, sample_weight=sample_weights_train)
mae, rmse, r2, rf_test_pred = evaluate_model(y_test_orig, rf.predict(X_test_scaled))
results['RandomForest'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
print(f'RandomForest | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')

---
## 7. Optuna MLP Optimization

Wide search space over architecture depth, width, normalization, dropout,
learning rate, weight decay, and Huber delta.

Uses **MedianPruner** to kill unpromising trials early.

In [ ]:
def create_mlp(trial, input_dim):
    """Build MLP model with Optuna-suggested hyperparameters."""
    n_layers = trial.suggest_int('n_layers', 2, 5)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    use_layernorm = trial.suggest_categorical('use_layernorm', [True, False])
    huber_delta = trial.suggest_float('huber_delta', 0.5, 5.0, step=0.5)
    lr = trial.suggest_float('lr', 1e-5, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for i in range(n_layers):
        units = trial.suggest_int(f'units_{i}', 64, 512, step=64)
        model.add(Dense(units, activation='relu'))
        if use_layernorm:
            model.add(LayerNormalization())
        # Reduce dropout for later layers
        model.add(Dropout(dropout_rate * (0.5 if i >= n_layers - 1 else 1.0)))

    model.add(Dense(1))

    opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    model.compile(optimizer=opt, loss=losses.Huber(delta=huber_delta), metrics=['mae'])
    return model


def objective_mlp(trial):
    """Optuna objective for MLP."""
    tf.keras.backend.clear_session()

    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    model = create_mlp(trial, X_train_scaled.shape[1])

    cb = [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True, min_delta=1e-4
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=7, min_lr=1e-6
        ),
        TFKerasPruningCallback(trial, 'val_loss'),
    ]

    model.fit(
        X_train_scaled, y_train_sqrt,
        sample_weight=sample_weights_train,
        validation_data=(X_val_scaled, y_val_sqrt),
        epochs=200, batch_size=batch_size,
        callbacks=cb, verbose=0
    )

    val_pred_sqrt = model.predict(X_val_scaled, verbose=0).flatten()
    val_pred_orig = np.clip(val_pred_sqrt, 0, None) ** 2
    val_rmse = np.sqrt(mean_squared_error(y_val_orig, val_pred_orig))
    return val_rmse

In [ ]:
# Run MLP optimization
mlp_study = optuna.create_study(
    direction='minimize',
    study_name='mlp_wr',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15),
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

print('Starting MLP Optuna optimization (100 trials)...')
print('This may take 30-60 minutes depending on hardware.\n')

mlp_study.optimize(objective_mlp, n_trials=100, show_progress_bar=True)

print(f'\nBest MLP trial:')
print(f'  Val RMSE: {mlp_study.best_value:.4f}')
print(f'  Params: {json.dumps(mlp_study.best_params, indent=2)}')

In [ ]:
# Train best MLP with full epochs
print('Training best MLP model with extended epochs...')
tf.keras.backend.clear_session()

best_p = mlp_study.best_params
n_layers = best_p['n_layers']
use_ln = best_p['use_layernorm']
dropout = best_p['dropout']

model_mlp = keras.Sequential()
model_mlp.add(layers.Input(shape=(X_train_scaled.shape[1],)))
for i in range(n_layers):
    model_mlp.add(Dense(best_p[f'units_{i}'], activation='relu'))
    if use_ln:
        model_mlp.add(LayerNormalization())
    model_mlp.add(Dropout(dropout * (0.5 if i >= n_layers - 1 else 1.0)))
model_mlp.add(Dense(1))

opt = optimizers.AdamW(learning_rate=best_p['lr'], weight_decay=best_p['weight_decay'])
model_mlp.compile(optimizer=opt, loss=losses.Huber(delta=best_p['huber_delta']), metrics=['mae'])

mlp_cb = [
    callbacks.ModelCheckpoint('mlp_optuna_best.keras', monitor='val_loss',
                              save_best_only=True, verbose=0),
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=10, min_lr=1e-6),
]

mlp_history = model_mlp.fit(
    X_train_scaled, y_train_sqrt,
    sample_weight=sample_weights_train,
    validation_data=(X_val_scaled, y_val_sqrt),
    epochs=500, batch_size=best_p['batch_size'],
    callbacks=mlp_cb, verbose=1
)

# Evaluate
mae, rmse, r2, mlp_test_pred = evaluate_model(y_test_orig, model_mlp.predict(X_test_scaled, verbose=0).flatten())
results['MLP (Optuna)'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
mlp_val_pred = model_mlp.predict(X_val_scaled, verbose=0).flatten()
print(f'\nMLP (Optuna) | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(mlp_history.history['loss'], label='Train')
axes[0].plot(mlp_history.history['val_loss'], label='Val')
axes[0].set_title('MLP Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(mlp_history.history['mae'], label='Train')
axes[1].plot(mlp_history.history['val_mae'], label='Val')
axes[1].set_title('MLP MAE (sqrt scale)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

---
## 8. Optuna RNN Optimization

Searches over:
- **Cell type:** LSTM vs GRU
- **Bidirectional:** yes/no
- **Depth:** 1-4 recurrent layers
- **Width:** 64-512 units per layer
- **Attention:** multi-head self-attention (yes/no, 2/4/8 heads)
- **Normalization:** LayerNorm vs BatchNorm vs none
- **Dense head:** 1-3 layers, 32-256 units
- **Sequence length:** 8, 12, 16, 24
- **Training:** LR, weight decay, dropout, Huber delta, batch size

Uses **MedianPruner** for early stopping of unpromising trials.

In [ ]:
def create_rnn(trial, seq_len, n_features):
    """Build RNN model with Optuna-suggested hyperparameters."""
    rnn_type = trial.suggest_categorical('rnn_type', ['LSTM', 'GRU'])
    bidirectional = trial.suggest_categorical('bidirectional', [True, False])
    n_rnn_layers = trial.suggest_int('n_rnn_layers', 1, 4)
    use_attention = trial.suggest_categorical('use_attention', [True, False])
    norm_type = trial.suggest_categorical('norm_type', ['layer', 'batch', 'none'])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    huber_delta = trial.suggest_float('huber_delta', 0.5, 5.0, step=0.5)
    lr = trial.suggest_float('lr', 1e-5, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

    RNNCell = LSTM if rnn_type == 'LSTM' else GRU

    inp = Input(shape=(seq_len, n_features))
    x = inp

    for i in range(n_rnn_layers):
        units = trial.suggest_int(f'rnn_units_{i}', 64, 512, step=64)
        return_seq = (i < n_rnn_layers - 1) or use_attention
        rnn_layer = RNNCell(units, return_sequences=return_seq)

        if bidirectional:
            x = Bidirectional(rnn_layer)(x)
        else:
            x = rnn_layer(x)

        if norm_type == 'layer':
            x = LayerNormalization()(x)
        elif norm_type == 'batch':
            x = BatchNormalization()(x)

        x = Dropout(dropout)(x)

    if use_attention:
        n_heads = trial.suggest_categorical('n_heads', [2, 4, 8])
        key_dim = trial.suggest_categorical('key_dim', [16, 32])
        x = MultiHeadAttention(num_heads=n_heads, key_dim=key_dim)(x, x)
        x = GlobalAveragePooling1D()(x)

    # Dense head
    n_dense = trial.suggest_int('n_dense_layers', 1, 3)
    for j in range(n_dense):
        dense_units = trial.suggest_int(f'dense_units_{j}', 32, 256, step=32)
        x = Dense(dense_units, activation='relu')(x)
        if norm_type == 'layer':
            x = LayerNormalization()(x)
        x = Dropout(dropout * 0.5)(x)

    out = Dense(1)(x)
    model = Model(inp, out)

    opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    model.compile(optimizer=opt, loss=losses.Huber(delta=huber_delta), metrics=['mae'])
    return model


def objective_rnn(trial):
    """Optuna objective for RNN."""
    tf.keras.backend.clear_session()

    seq_len = trial.suggest_categorical('seq_len', SEQ_LENGTHS)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    data = seq_data[seq_len]
    X_tr, y_tr_sqrt, y_tr_orig = data['train']
    X_va, y_va_sqrt, y_va_orig = data['val']

    # Sample weights for this sequence length
    sw = seq_weights[seq_len]

    model = create_rnn(trial, seq_len, n_features)

    cb = [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=12, restore_best_weights=True, min_delta=1e-4
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=5, min_lr=1e-6
        ),
        TFKerasPruningCallback(trial, 'val_loss'),
    ]

    model.fit(
        X_tr, y_tr_sqrt,
        sample_weight=sw,
        validation_data=(X_va, y_va_sqrt),
        epochs=150, batch_size=batch_size,
        callbacks=cb, verbose=0
    )

    val_pred_sqrt = model.predict(X_va, verbose=0).flatten()
    val_pred_orig = np.clip(val_pred_sqrt, 0, None) ** 2
    val_rmse = np.sqrt(mean_squared_error(y_va_orig, val_pred_orig))
    return val_rmse

In [ ]:
# Run RNN optimization
rnn_study = optuna.create_study(
    direction='minimize',
    study_name='rnn_wr',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10),
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

print('Starting RNN Optuna optimization (100 trials)...')
print('This may take 2-4 hours depending on hardware.\n')

rnn_study.optimize(objective_rnn, n_trials=100, show_progress_bar=True)

print(f'\nBest RNN trial:')
print(f'  Val RMSE: {rnn_study.best_value:.4f}')
print(f'  Params: {json.dumps(rnn_study.best_params, indent=2)}')

In [ ]:
# Train best RNN with extended epochs
print('Training best RNN model with extended epochs...')
tf.keras.backend.clear_session()

bp = rnn_study.best_params
best_seq_len = bp['seq_len']
data = seq_data[best_seq_len]
X_tr, y_tr_sqrt, y_tr_orig = data['train']
X_va, y_va_sqrt, y_va_orig = data['val']
X_te, y_te_sqrt, y_te_orig = data['test']
sw = seq_weights[best_seq_len]

# Reconstruct best model
best_trial = rnn_study.best_trial
model_rnn = create_rnn(best_trial, best_seq_len, n_features)
model_rnn.summary()

rnn_cb = [
    callbacks.ModelCheckpoint('rnn_optuna_best.keras', monitor='val_loss',
                              save_best_only=True, verbose=0),
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=10, min_lr=1e-6),
]

rnn_history = model_rnn.fit(
    X_tr, y_tr_sqrt,
    sample_weight=sw,
    validation_data=(X_va, y_va_sqrt),
    epochs=500, batch_size=bp['batch_size'],
    callbacks=rnn_cb, verbose=1
)

# Evaluate on test
mae, rmse, r2, rnn_test_pred = evaluate_model(y_te_orig, model_rnn.predict(X_te, verbose=0).flatten())
results['RNN (Optuna)'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
rnn_val_pred = model_rnn.predict(X_va, verbose=0).flatten()
print(f'\nRNN (Optuna) | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')
print(f'Best seq_len={best_seq_len}, type={bp["rnn_type"]}, bi={bp["bidirectional"]}')

# Plot training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(rnn_history.history['loss'], label='Train')
axes[0].plot(rnn_history.history['val_loss'], label='Val')
axes[0].set_title('RNN Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(rnn_history.history['mae'], label='Train')
axes[1].plot(rnn_history.history['val_mae'], label='Val')
axes[1].set_title('RNN MAE (sqrt scale)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

---
## 9. Optuna TCN (Temporal Convolutional Network)

TCN uses **dilated causal convolutions** to capture temporal patterns.
Often faster to train than RNNs and can model longer-range dependencies
efficiently through exponentially increasing dilation rates.

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    """Single TCN residual block with causal dilated convolution."""
    shortcut = x

    x = Conv1D(filters, kernel_size, dilation_rate=dilation_rate,
               padding='causal', activation='relu')(x)
    x = LayerNormalization()(x)
    x = Dropout(dropout_rate)(x)

    x = Conv1D(filters, kernel_size, dilation_rate=dilation_rate,
               padding='causal', activation='relu')(x)
    x = LayerNormalization()(x)
    x = Dropout(dropout_rate)(x)

    # Match dimensions for residual connection
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding='same')(shortcut)

    x = Add()([shortcut, x])
    x = Activation('relu')(x)
    return x


def create_tcn(trial, seq_len, n_features):
    """Build TCN model with Optuna-suggested hyperparameters."""
    n_blocks = trial.suggest_int('n_blocks', 2, 5)
    filters = trial.suggest_int('filters', 64, 256, step=64)
    kernel_size = trial.suggest_categorical('kernel_size', [2, 3, 4])
    dropout = trial.suggest_float('tcn_dropout', 0.1, 0.4, step=0.05)
    huber_delta = trial.suggest_float('tcn_huber_delta', 0.5, 5.0, step=0.5)
    lr = trial.suggest_float('tcn_lr', 1e-5, 5e-3, log=True)
    wd = trial.suggest_float('tcn_wd', 1e-6, 1e-2, log=True)

    inp = Input(shape=(seq_len, n_features))
    x = inp

    for i in range(n_blocks):
        dilation_rate = 2 ** i
        x = residual_block(x, filters, kernel_size, dilation_rate, dropout)

    # Aggregate temporal dimension
    use_attention = trial.suggest_categorical('tcn_attention', [True, False])
    if use_attention:
        n_heads = trial.suggest_categorical('tcn_n_heads', [2, 4])
        x = MultiHeadAttention(num_heads=n_heads, key_dim=32)(x, x)
    x = GlobalAveragePooling1D()(x)

    # Dense head
    dense_units = trial.suggest_int('tcn_dense', 32, 256, step=32)
    x = Dense(dense_units, activation='relu')(x)
    x = Dropout(dropout * 0.5)(x)
    out = Dense(1)(x)

    model = Model(inp, out)
    opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    model.compile(optimizer=opt, loss=losses.Huber(delta=huber_delta), metrics=['mae'])
    return model


def objective_tcn(trial):
    """Optuna objective for TCN."""
    tf.keras.backend.clear_session()

    seq_len = trial.suggest_categorical('tcn_seq_len', SEQ_LENGTHS)
    batch_size = trial.suggest_categorical('tcn_batch', [32, 64])

    data = seq_data[seq_len]
    X_tr, y_tr_sqrt, y_tr_orig = data['train']
    X_va, y_va_sqrt, y_va_orig = data['val']
    sw = seq_weights[seq_len]

    model = create_tcn(trial, seq_len, n_features)

    cb = [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=12, restore_best_weights=True, min_delta=1e-4
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=5, min_lr=1e-6
        ),
        TFKerasPruningCallback(trial, 'val_loss'),
    ]

    model.fit(
        X_tr, y_tr_sqrt, sample_weight=sw,
        validation_data=(X_va, y_va_sqrt),
        epochs=150, batch_size=batch_size,
        callbacks=cb, verbose=0
    )

    val_pred_sqrt = model.predict(X_va, verbose=0).flatten()
    val_pred_orig = np.clip(val_pred_sqrt, 0, None) ** 2
    val_rmse = np.sqrt(mean_squared_error(y_va_orig, val_pred_orig))
    return val_rmse

In [ ]:
# Run TCN optimization
tcn_study = optuna.create_study(
    direction='minimize',
    study_name='tcn_wr',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10),
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

print('Starting TCN Optuna optimization (80 trials)...')
print('This may take 1-3 hours depending on hardware.\n')

tcn_study.optimize(objective_tcn, n_trials=80, show_progress_bar=True)

print(f'\nBest TCN trial:')
print(f'  Val RMSE: {tcn_study.best_value:.4f}')
print(f'  Params: {json.dumps(tcn_study.best_params, indent=2)}')

In [ ]:
# Train best TCN with extended epochs
print('Training best TCN model with extended epochs...')
tf.keras.backend.clear_session()

bp_tcn = tcn_study.best_params
tcn_seq_len = bp_tcn['tcn_seq_len']
data = seq_data[tcn_seq_len]
X_tr, y_tr_sqrt, y_tr_orig = data['train']
X_va, y_va_sqrt, y_va_orig = data['val']
X_te, y_te_sqrt, y_te_orig = data['test']
sw = seq_weights[tcn_seq_len]

best_tcn_trial = tcn_study.best_trial
model_tcn = create_tcn(best_tcn_trial, tcn_seq_len, n_features)
model_tcn.summary()

tcn_cb = [
    callbacks.ModelCheckpoint('tcn_optuna_best.keras', monitor='val_loss',
                              save_best_only=True, verbose=0),
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=10, min_lr=1e-6),
]

tcn_history = model_tcn.fit(
    X_tr, y_tr_sqrt, sample_weight=sw,
    validation_data=(X_va, y_va_sqrt),
    epochs=500, batch_size=bp_tcn['tcn_batch'],
    callbacks=tcn_cb, verbose=1
)

mae, rmse, r2, tcn_test_pred = evaluate_model(y_te_orig, model_tcn.predict(X_te, verbose=0).flatten())
results['TCN (Optuna)'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
tcn_val_pred = model_tcn.predict(X_va, verbose=0).flatten()
print(f'\nTCN (Optuna) | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')
print(f'Best seq_len={tcn_seq_len}, blocks={bp_tcn["n_blocks"]}, filters={bp_tcn["filters"]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(tcn_history.history['loss'], label='Train')
axes[0].plot(tcn_history.history['val_loss'], label='Val')
axes[0].set_title('TCN Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(tcn_history.history['mae'], label='Train')
axes[1].plot(tcn_history.history['val_mae'], label='Val')
axes[1].set_title('TCN MAE (sqrt scale)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

---
## 10. Ensemble

Weighted combination of the best models.
Optimize weights on validation set to minimize RMSE.

In [ ]:
# Collect validation predictions (all on original scale)
# For flat models: predict on val set directly
# For sequence models: use their respective val splits

# XGBoost & LightGBM val predictions (already in sqrt scale)
xgb_val_orig = np.clip(xgb_val_pred, 0, None) ** 2
lgb_val_orig = np.clip(lgb_val_pred, 0, None) ** 2

# MLP val prediction (already in sqrt scale)
mlp_val_orig = np.clip(mlp_val_pred, 0, None) ** 2

# For RNN and TCN, val predictions are on their respective sequence-based val sets
# which may differ from flat val set. We need to evaluate ensemble on the common test set.

# === Test set ensemble ===
# All test predictions are already computed and stored in *_test_pred (original scale)
# xgb_test_pred, lgb_test_pred, mlp_test_pred: from flat test set
# rnn_test_pred, tcn_test_pred: from sequence test set (different indices!)

# For a proper ensemble, we need predictions on the SAME test samples.
# Solution: use flat models for the ensemble (XGBoost, LightGBM, MLP)
# since they share the same test indices.

print('=== Flat Model Ensemble (same test set) ===')
flat_preds = {
    'XGBoost': xgb_test_pred,
    'LightGBM': lgb_test_pred,
    'MLP': mlp_test_pred,
}
flat_val_preds = {
    'XGBoost': xgb_val_orig,
    'LightGBM': lgb_val_orig,
    'MLP': mlp_val_orig,
}

# Grid search for best weights on validation set
best_w, best_rmse_ens = None, float('inf')

for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < -0.01:
            continue
        w3 = max(w3, 0)

        ens_pred = w1 * flat_val_preds['XGBoost'] + w2 * flat_val_preds['LightGBM'] + w3 * flat_val_preds['MLP']
        rmse = np.sqrt(mean_squared_error(y_val_orig, ens_pred))

        if rmse < best_rmse_ens:
            best_rmse_ens = rmse
            best_w = (w1, w2, w3)

print(f'Best ensemble weights: XGB={best_w[0]:.2f}, LGB={best_w[1]:.2f}, MLP={best_w[2]:.2f}')
print(f'Val RMSE: {best_rmse_ens:.4f}')

# Apply to test set
ens_test = best_w[0] * flat_preds['XGBoost'] + best_w[1] * flat_preds['LightGBM'] + best_w[2] * flat_preds['MLP']
ens_mae = mean_absolute_error(y_test_orig, ens_test)
ens_rmse = np.sqrt(mean_squared_error(y_test_orig, ens_test))
ens_r2 = r2_score(y_test_orig, ens_test)
results['Ensemble (Flat)'] = {'MAE': ens_mae, 'RMSE': ens_rmse, 'R2': ens_r2}
print(f'\nEnsemble Test | MAE={ens_mae:.2f}  RMSE={ens_rmse:.2f}  R2={ens_r2:.4f}')

---
## 11. Final Comparison & Visualization

All models compared on the same test set (2024-2025 seasons).
Metrics on original receiving yards scale.

In [ ]:
# Results table
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('R2', ascending=False)
results_df.index.name = 'Model'
print('\n' + '='*70)
print('FINAL RESULTS — Test Set (2024-2025)')
print('='*70)
print(results_df.round(4).to_string())
print('='*70)

# Save results
results_df.to_csv('../results/career_rnn_optuna_final_results.csv')
print('\nResults saved to results/career_rnn_optuna_final_results.csv')

In [ ]:
# --- Bar chart: RMSE comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(results_df)))

axes[0].barh(results_df.index, results_df['MAE'], color=colors)
axes[0].set_xlabel('MAE (yards)')
axes[0].set_title('Mean Absolute Error')
axes[0].invert_yaxis()

axes[1].barh(results_df.index, results_df['RMSE'], color=colors)
axes[1].set_xlabel('RMSE (yards)')
axes[1].set_title('Root Mean Squared Error')
axes[1].invert_yaxis()

r2_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(results_df)))
axes[2].barh(results_df.index, results_df['R2'], color=r2_colors[::-1])
axes[2].set_xlabel('R2')
axes[2].set_title('R-Squared')
axes[2].invert_yaxis()

plt.suptitle('Model Comparison — WR Receiving Yards Prediction', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/career_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/career_model_comparison.png')

In [ ]:
# --- Diagnostic scatter plots for top 3 models ---
top3 = results_df.head(3).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, model_name in enumerate(top3):
    if 'RNN' in model_name:
        data_te = seq_data[best_seq_len]['test']
        y_true = data_te[2]
        y_pred = rnn_test_pred
    elif 'TCN' in model_name:
        data_te = seq_data[tcn_seq_len]['test']
        y_true = data_te[2]
        y_pred = tcn_test_pred
    elif 'Ensemble' in model_name:
        y_true = y_test_orig
        y_pred = ens_test
    elif 'MLP' in model_name:
        y_true = y_test_orig
        y_pred = mlp_test_pred
    elif 'XGBoost' in model_name:
        y_true = y_test_orig
        y_pred = xgb_test_pred
    elif 'LightGBM' in model_name:
        y_true = y_test_orig
        y_pred = lgb_test_pred
    else:
        y_true = y_test_orig
        y_pred = rf_test_pred

    # Scatter: predicted vs actual
    axes[0, idx].scatter(y_true, y_pred, alpha=0.3, s=10)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[0, idx].plot(lims, lims, 'r--', linewidth=1)
    axes[0, idx].set_xlabel('Actual')
    axes[0, idx].set_ylabel('Predicted')
    axes[0, idx].set_title(f'{model_name}\nPredicted vs Actual')

    # Residuals
    residuals = y_pred - y_true
    axes[1, idx].scatter(y_pred, residuals, alpha=0.3, s=10)
    axes[1, idx].axhline(y=0, color='r', linestyle='--', linewidth=1)
    axes[1, idx].set_xlabel('Predicted')
    axes[1, idx].set_ylabel('Residual')
    axes[1, idx].set_title(f'{model_name}\nResiduals')

plt.suptitle('Top 3 Models — Diagnostic Plots', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/career_top3_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Optuna optimization history ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MLP study
mlp_vals = [t.value for t in mlp_study.trials if t.value is not None]
axes[0].plot(mlp_vals, 'o-', markersize=3, alpha=0.7)
axes[0].axhline(y=mlp_study.best_value, color='r', linestyle='--', label=f'Best: {mlp_study.best_value:.3f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val RMSE'); axes[0].set_title('MLP Optuna History')
axes[0].legend()

# RNN study
rnn_vals = [t.value for t in rnn_study.trials if t.value is not None]
axes[1].plot(rnn_vals, 'o-', markersize=3, alpha=0.7)
axes[1].axhline(y=rnn_study.best_value, color='r', linestyle='--', label=f'Best: {rnn_study.best_value:.3f}')
axes[1].set_xlabel('Trial'); axes[1].set_ylabel('Val RMSE'); axes[1].set_title('RNN Optuna History')
axes[1].legend()

# TCN study
tcn_vals = [t.value for t in tcn_study.trials if t.value is not None]
axes[2].plot(tcn_vals, 'o-', markersize=3, alpha=0.7)
axes[2].axhline(y=tcn_study.best_value, color='r', linestyle='--', label=f'Best: {tcn_study.best_value:.3f}')
axes[2].set_xlabel('Trial'); axes[2].set_ylabel('Val RMSE'); axes[2].set_title('TCN Optuna History')
axes[2].legend()

plt.suptitle('Optuna Optimization Progress', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/career_optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Hyperparameter importance (Optuna) ---
from optuna.importance import get_param_importances

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, study, name in [(axes[0], mlp_study, 'MLP'),
                          (axes[1], rnn_study, 'RNN'),
                          (axes[2], tcn_study, 'TCN')]:
    try:
        importances = get_param_importances(study)
        top_k = dict(list(importances.items())[:10])
        ax.barh(list(top_k.keys())[::-1], list(top_k.values())[::-1])
        ax.set_title(f'{name} — HP Importance')
        ax.set_xlabel('Importance')
    except Exception as e:
        ax.text(0.5, 0.5, f'Could not compute\nimportance: {e}',
                transform=ax.transAxes, ha='center', va='center')
        ax.set_title(f'{name} — HP Importance')

plt.tight_layout()
plt.savefig('../results/career_hp_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Save Optuna study summaries ---
summary = {
    'MLP': {
        'best_val_rmse': mlp_study.best_value,
        'best_params': mlp_study.best_params,
        'n_trials': len(mlp_study.trials),
    },
    'RNN': {
        'best_val_rmse': rnn_study.best_value,
        'best_params': rnn_study.best_params,
        'n_trials': len(rnn_study.trials),
    },
    'TCN': {
        'best_val_rmse': tcn_study.best_value,
        'best_params': tcn_study.best_params,
        'n_trials': len(tcn_study.trials),
    },
}

with open('../results/career_optuna_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('Optuna summaries saved to results/career_optuna_summary.json')
print('\n=== DONE ===')
print(f'Best overall model: {results_df.index[0]} with R2={results_df["R2"].iloc[0]:.4f}')